[source](https://scicomp.stackexchange.com/questions/37336/solving-numerically-the-1d-kuramoto-sivashinsky-equation-using-spectral-methods)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import cm

In [ ]:
nu = 1
L = 100 
nx = 1024

t0 = 0 
tN = 200
dt = 0.05
nt = int((tN - t0) / 0.05)

# wave number mesh
k = np.arange(-nx/2, nx/2, 1)

t = np.linspace(start=t0, stop=tN, num=nt)
x = np.linspace(start=0, stop=L, num=nx)

# solution mesh in real space
u = np.ones((nx, nt))
# solution mesh in Fourier space
u_hat = np.ones((nx, nt), dtype=complex)

u_hat2 = np.ones((nx, nt), dtype=complex)

# initial condition 
u0 = np.cos((2 * np.pi * x) / L) + 0.1 * np.cos((4 * np.pi * x) / L)

# Fourier transform of initial condition
u0_hat = (1 / nx) * np.fft.fftshift(np.fft.fft(u0))

u0_hat2 = (1 / nx) * np.fft.fftshift(np.fft.fft(u0**2))

# set initial condition in real and Fourier mesh
u[:,0] = u0
u_hat[:,0] = u0_hat

u_hat2[:,0] = u0_hat2


In [ ]:

# Fourier Transform of the linear operator
FL = (((2 * np.pi) / L) * k) ** 2 - nu * (((2 * np.pi) / L) * k) ** 4
# Fourier Transform of the non-linear operator
FN = - (1 / 2) * ((1j) * ((2 * np.pi) / L) * k)

# resolve EDP in Fourier space
for j in range(0,nt-1):
  uhat_current = u_hat[:,j]
  uhat_current2 = u_hat2[:,j]
  if j == 0:
    uhat_last = u_hat[:,0]
    uhat_last2 = u_hat2[:,0]
  else:
    uhat_last = u_hat[:,j-1]
    uhat_last2 = u_hat2[:,j-1]
  
  # compute solution in Fourier space through a finite difference method
  # Cranck-Nicholson + Adam 
  u_hat[:,j+1] = (1 / (1 - (dt / 2) * FL)) * ( (1 + (dt / 2) * FL) * uhat_current + ( ((3 / 2) * FN) * (uhat_current2) - ((1 / 2) * FN) * (uhat_last2) ) * dt )
  # go back in real space
  u[:,j+1] = np.real(nx * np.fft.ifft(np.fft.ifftshift(u_hat[:,j+1])))
  u_hat2[:,j+1] = (1 / nx) * np.fft.fftshift(np.fft.fft(u[:,j+1]**2))


In [ ]:
# plot the result
fig, ax = plt.subplots(figsize=(10,8))

xx, tt = np.meshgrid(x, t)
cs = ax.contourf(xx, tt, u.T, cmap=cm.jet)
fig.colorbar(cs)

ax.set_xlabel("x")
ax.set_ylabel("t")
ax.set_title(f"Kuramoto-Sivashinsky: L = {L}, nu = {nu}")

In [ ]:
from IPython.display import clear_output
import time

def visualize(x, u, frame_skip=100, max_frames=20):
    n_time = u.shape[1]

    total_frames = min(n_time // frame_skip, max_frames)
    frames_to_show = np.linspace(0, n_time-1, total_frames, dtype=int)

    for frame in frames_to_show:
        clear_output(wait=True)

        plt.figure(figsize=(10, 6))
        plt.plot(x, u[:, frame], 'b-', linewidth=2)
        plt.xlabel('Position (x)')
        plt.ylabel('Amplitude (u)')
        plt.title(f'Frame {frame} ({(frame/(n_time-1)*100):.1f}%)')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        time.sleep(0.5)


In [ ]:
#visualize(x, u, 10, 200)

In [ ]:
data = u.T
np.save("1dks.npy", data)

In [ ]:
from pyshred import DataManager, SHRED, SHREDEngine

import pyshred
pyshred.set_default_device(device="cuda")

manager = DataManager(
    lags=500,
    train_size=0.7,
    val_size=0.15,
    test_size=0.15
)

In [ ]:
manager.add_data(
    data=data,
    id="1dks",
    random=10,
    compress=20
)

In [ ]:
train_dataset, val_dataset, test_dataset= manager.prepare()

In [ ]:
shred = SHRED(
    sequence_model="LSTM",
    decoder_model="MLP",
    latent_forecaster="LSTM_Forecaster"
)

In [ ]:
val_errors = shred.fit(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    lr=3e-4,
    num_epochs=500,
    verbose=True
)

print('val_errors:', val_errors)


In [ ]:
train_mse = shred.evaluate(dataset=train_dataset)
val_mse = shred.evaluate(dataset=val_dataset)
test_mse = shred.evaluate(dataset=test_dataset)
print(f"Train MSE: {train_mse:.3f}")
print(f"Val   MSE: {val_mse:.3f}")
print(f"Test  MSE: {test_mse:.3f}")


In [ ]:
engine = SHREDEngine(manager, shred)


In [ ]:
# obtain latent space of test sensor measurements
test_latent_from_sensors = engine.sensor_to_latent(manager.test_sensor_measurements)


In [ ]:
# generate latent states from validation sensor measurements
val_latents = engine.sensor_to_latent(manager.val_sensor_measurements)

# seed the forecaster with the final `seed_length` latent states from validation
init_latents = val_latents[-shred.latent_forecaster.seed_length:] # seed forecaster with final lag timesteps of latent space from val

# set forecast horizon to match the length of the test dataset
h = len(manager.test_sensor_measurements)

# forecast latent states for the test horizon
test_latent_from_forecaster = engine.forecast_latent(h=h, init_latents=init_latents)

In [ ]:
# decode latent space generated from sensor measurements (generated using engine.sensor_to_latent())
test_reconstruction = engine.decode(test_latent_from_sensors)

# decode latent space generated by the latent forecaster (generated using engine.forecast_latent())
test_forecast = engine.decode(test_latent_from_forecaster)

In [ ]:
test_reconstruction["1dks"].shape, test_forecast["1dks"].shape

In [ ]:
u.shape

In [ ]:
data_to_plot = test_reconstruction["1dks"].T
fig, ax = plt.subplots(figsize=(10,8))

xx, tt = np.meshgrid(x, t[-data_to_plot.shape[1]:])
levels = np.arange(-3, 3, 0.01)
cs = ax.contourf(xx, tt, data_to_plot.T, cmap=cm.inferno)
fig.colorbar(cs)

ax.set_xlabel("x")
ax.set_ylabel("t")
ax.set_title(f"Reconstruction mode: L = {L}, nu = {nu}")

In [ ]:
data_to_plot = test_reconstruction["1dks"].T
fig, ax = plt.subplots(figsize=(10,8))

xx, tt = np.meshgrid(x, t[-data_to_plot.shape[1]:])
levels = np.arange(-3, 3, 0.01)
cs = ax.contourf(xx, tt, np.sqrt((u.T[-data_to_plot.shape[1]:] - data_to_plot.T)**2), cmap=cm.inferno)
fig.colorbar(cs)

ax.set_xlabel("x")
ax.set_ylabel("t")
ax.set_title(f"L2 norm between rec and true: L = {L}, nu = {nu}")

TO-DO: plot relative error

In [ ]:
eps = 1e-12
rel_err = abs_err / (np.abs(X_true) + eps) * 100

In [ ]:
#visualize(x, test_forecast["1dks"].T, 10, 200)